# Ранжирование и процентная динамика

# Задание
Загрузите файл в датафреймы _Pandas_ и назовите его _coderun_. 

Создайте в датафрейме _coderun_ столбец _rnk_, который ведет себя как оконная функция **RANK** в **SQL** - ранжирует строки по заданным правилам. Правила такие:


- ранжирование строк должно быть отдельным в рамках каждого пользователя

- ранжирование происходит на основании номера задачи и даты-времени ее решения

- если пользователь и задача совпадают, принимаем решение на основании даты-времени

- все сортировки берем по возрастанию

После этого создайте датафрейм _df_:


- оставьте только ранги, которые больше 3

- по всем рангам посчитайте количество уникальных пользователей, у которых встречался данный ранг (назовите столбец amount)

- столбец с рангом должен быть не в индексе

А в конце добавьте столбец _diff_, который покажет процентную динамику между текущим _amount_ и предыдущим. Возникшие в результате пропуске замените на 0. Значения округлите до 2 знака после запятой.

In [1]:
import pandas as pd

In [2]:
coderun = pd.read_csv('D:/GitHub_projects/Simulative_course/data/itresume-coderun.csv', encoding='1251')

In [3]:
coderun.head()

,id,created_at,problem_id,user_id,language_id
0,1,2021-04-07 06:06:20.000,13,10,3
1,2,2021-03-31 07:10:06.000,15,13,3
2,3,2021-04-04 14:55:26.000,1,6,3
3,4,2021-03-29 21:24:51.000,26,4,3
4,5,2021-03-30 11:29:12.000,21,18,3


In [22]:
coderun['rnk'] = coderun.sort_values(['user_id', 'problem_id', 'created_at']).groupby('user_id').cumcount() + 1
coderun

,id,created_at,problem_id,user_id,language_id,rnk
0,1,2021-04-07 06:06:20.000,13,10,3,2
1,2,2021-03-31 07:10:06.000,15,13,3,1
2,3,2021-04-04 14:55:26.000,1,6,3,1
3,4,2021-03-29 21:24:51.000,26,4,3,1
4,5,2021-03-30 11:29:12.000,21,18,3,2
...,...,...,...,...,...,...
55717,55718,2022-05-17 09:00:51.743,101,171,2,283
55718,55719,2022-05-17 09:01:04.210,101,171,2,284
55719,55720,2022-05-17 09:02:06.203,101,171,2,285
55720,55721,2022-05-17 09:03:45.265,102,171,2,286


In [23]:
df = coderun[coderun['rnk'] > 3]. \
    groupby('rnk'). \
    apply(lambda x: pd.Series(data=x['user_id'].nunique(), index=['amount']), include_groups=False). \
    reset_index()
df['diff'] = (df['amount'].pct_change()*100).round(2).fillna(0)
df

,rnk,amount,diff
0,4,660,0.00
1,5,625,-5.30
2,6,595,-4.80
3,7,569,-4.37
4,8,543,-4.57
...,...,...,...
1458,1462,1,0.00
1459,1463,1,0.00
1460,1464,1,0.00
1461,1465,1,0.00
